# Table of Contents
* [Section 1: Imports & Configuration](#imports)
* [Section 2: Extracting Words With Their Timecodes](#extract-words)
* [Section 3: Mapping Words To Phonemes](#map-words)
* [Section 4: Rhyme Unit Extraction](#extract-rhyme-units)
* [Section 5: Pairwise Rhyme Similarity](#pairwise-similarity)
* [Section 6: Words Clustering](#clustering)
* [Section 7: HTML Generation](#html-generation)
* [Section 8: Syllable Engine (boundary-free) — primary view](#syllable-engine)

---
## Section 1: Imports & Configuration <a id="imports"></a>
---

In [1]:
import sys
sys.path.append("..")
print(sys.executable)

from python.rhyme_extraction import (
    extract_rhyme_unit, 
    extract_end_words, 
    extract_rhyme_candidates,
    extract_syllable_candidates
)

from python.similarity_engine import (
    rhyme_similarity,
    compute_similarity_pairs,
    build_similarity_matrix,
    compute_syllable_pairs,
    build_syllable_matrix
)
from python.clustering import (
    SIMILARITY_THRESHOLD,
    get_threshold,
    cluster_rhymes, 
    cluster_exact_rhymes
)
from python.debugging import debug_rhyme_analysis
from python.evaluation import evaluate_rhyme_detection
from python.html_generation import generate_rhyme_html, generate_syllable_html
from python.score_logging import log_scored_pairs

from praatio import textgrid

RHYME_MODE = "stressed"
DETECTION_MODE = "full_line"
# Options for RHYME_MODE:   "stressed" / "stressed_plus" / "entire_word"
# Options for DETECTION_MODE: "end_only" / "full_line"

INPUT_TXT_PATH = "input/current_input/input.txt"
OUTPUT_TEXTGRID_PATH = "output/current_output/input.TextGrid"

# Milestone 19a — passive pair-logging dataset
VERSE_ID = "eminem-load-the-clip"   # identifies the aligned verse for the M19 dataset
DATASET_PATH = "data/scored_pairs.jsonl"

# Milestone 13 — syllable engine clustering threshold (carried at 0.7, re-swept)
SYLLABLE_THRESHOLD = 0.7

C:\Users\Georgii\miniconda3\envs\mfa_env\python.exe


---
## Section 2: Extracting Words With Their Timecodes <a id="extract-words"></a>
---

In [2]:
tg = textgrid.openTextgrid(OUTPUT_TEXTGRID_PATH, includeEmptyIntervals=False)

word_tier = tg.getTier("words")
phone_tier = tg.getTier("phones")

for word in word_tier.entries:
    print(word)

Interval(start=0.0, end=0.43, label='but')
Interval(start=0.43, end=0.67, label='still')
Interval(start=0.67, end=0.77, label='as')
Interval(start=0.77, end=1.34, label='explosive')
Interval(start=1.34, end=1.51, label='with')
Interval(start=1.51, end=1.74, label='just')
Interval(start=1.74, end=1.89, label='load')
Interval(start=2.03, end=2.09, label='the')
Interval(start=2.09, end=2.25, label='clip')
Interval(start=2.25, end=2.4, label='two')
Interval(start=2.4, end=2.97, label='pistols')
Interval(start=2.97, end=3.08, label='on')
Interval(start=3.08, end=3.27, label='hip')
Interval(start=3.27, end=3.59, label='both')
Interval(start=3.59, end=3.62, label='are')
Interval(start=3.62, end=3.86, label='gripped')
Interval(start=3.86, end=3.92, label='i')
Interval(start=3.92, end=4.17, label='hold')
Interval(start=4.17, end=4.3, label='them')
Interval(start=4.3, end=4.48, label='at')
Interval(start=4.48, end=4.77, label='shoulder')
Interval(start=4.77, end=5.03, label='width')
Interval(sta

---
## Section 3: Mapping Words To Phonemes <a id="map-words"></a>
---

In [3]:
for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    phonemes = []
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    print(word_label + ":", phonemes)

# Populating the word_to_phonemes dictionary
word_to_phonemes = {}

for word in word_tier.entries:
    word_start, word_end, word_label = word.start, word.end, word.label
    
    phonemes = []
    
    for phone in phone_tier.entries:
        if phone.start >= word_start and phone.end <= word_end:
            phonemes.append(phone.label)
    
    word_label = word_label.lower()
    
    if word_label not in word_to_phonemes:
        word_to_phonemes[word_label] = phonemes

print(word_to_phonemes)

but: ['b', 'a', 't']
still: ['s', 'tʲ', 'i', 'ɫ']
as: ['a', 's']
explosive: ['ɛ', 'k', 's', 'p', 'l', 'o', 's', 'i', 'v']
with: ['ʋ', 'ɪ', 'd̪']
just: ['dʒ', 'ɐ', 's', 't']
load: ['l', 'əw', 'd']
the: ['ð', 'a']
clip: ['c', 'ʎ', 'i', 'p']
two: ['ʈ', 'ʉː']
pistols: ['p', 'ɪ', 's', 'ʈ', 'ə', 'l', 'z']
on: ['a', 'n']
hip: ['ç', 'i', 'p']
both: ['b', 'əw', 'θ']
are: ['a']
gripped: ['ɟ', 'ɹ', 'i', 'p', 't']
i: ['aj']
hold: ['h', 'əw', 'ɫ', 'd']
them: ['ð', 'ə', 'm']
at: ['a', 't']
shoulder: ['ʃ', 'əw', 'ɫ', 'd', 'ə']
width: ['ʋ', 'ɪ', 'd̪']
took: ['ʈ', 'ʊ', 'k']
a: ['a']
stab: ['s', 't', 'a', 'b']
in: ['i', 'n']
the: ['ð', 'a']
dark: ['d', 'a', 'k']
and: ['a', 'n']
broke: ['b', 'ɹ', 'əw', 'k']
the: ['ð', 'a']
tip: ['ʈ', 'ɪ', 'p']
of: ['ɒ', 'v']
my: ['m', 'ɑ']
knife: ['n', 'aj', 'f']
off: ['ɑ', 'f']
but: ['b', 'a', 't']
your: ['j', 'ɒ', 'ɹ']
throat: ['t̪', 'ɹ', 'oː', 'ʈ']
is: ['i', 'z']
slit: ['s', 'ʎ', 'i', 't']
cause: ['k', 'ɒ', 'z']
i'm: ['aj', 'm']
cutthroat: ['k', 'ə', 't̪', 'ɹ', 'oː', 

---
## Section 4: Rhyme Unit Extraction <a id="extract-rhyme-units"></a>
---

In [4]:
rhyme_candidates = extract_rhyme_candidates(
    INPUT_TXT_PATH,
    word_to_phonemes,
    rhyme_mode=RHYME_MODE,
    detection_mode=DETECTION_MODE
)

for r in rhyme_candidates:
    print(r)

{'line_index': 0, 'line_text': 'But still as explosive with, just load the clip', 'word_index': 0, 'end_word': 'but', 'phonemes': ['b', 'a', 't'], 'rhyme_unit': ['a', 't'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'But still as explosive with, just load the clip', 'word_index': 1, 'end_word': 'still', 'phonemes': ['s', 'tʲ', 'i', 'ɫ'], 'rhyme_unit': ['i', 'ɫ'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'But still as explosive with, just load the clip', 'word_index': 2, 'end_word': 'as', 'phonemes': ['a', 's'], 'rhyme_unit': ['a', 's'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'But still as explosive with, just load the clip', 'word_index': 3, 'end_word': 'explosive', 'phonemes': ['ɛ', 'k', 's', 'p', 'l', 'o', 's', 'i', 'v'], 'rhyme_unit': ['i', 'v'], 'is_line_end': False}
{'line_index': 0, 'line_text': 'But still as explosive with, just load the clip', 'word_index': 4, 'end_word': 'with', 'phonemes': ['ʋ', 'ɪ', 'd̪'], 'rhyme_unit': ['ɪ', 'd̪'], 'is_li

---
## Section 5: Pairwise Rhyme Similarity <a id="pairwise-similarity"></a>
---

In [5]:
similarity_pairs = compute_similarity_pairs(rhyme_candidates, rhyme_mode=RHYME_MODE)

for pair in similarity_pairs:
    print(pair)

# Building a similarity matrix
similarity_matrix = build_similarity_matrix(rhyme_candidates, similarity_pairs)

for w1 in similarity_matrix:
    print(w1, similarity_matrix[w1])

{'wordA': 'but', 'wordB': 'still', 'similarity': 0.02499999999999991}
{'wordA': 'but', 'wordB': 'as', 'similarity': 0.8125}
{'wordA': 'but', 'wordB': 'explosive', 'similarity': 0.02499999999999991}
{'wordA': 'but', 'wordB': 'with', 'similarity': 0.17500000000000004}
{'wordA': 'but', 'wordB': 'just', 'similarity': 0.5916666666666666}
{'wordA': 'but', 'wordB': 'load', 'similarity': 0.6499999999999999}
{'wordA': 'but', 'wordB': 'the', 'similarity': 0.8875}
{'wordA': 'but', 'wordB': 'clip', 'similarity': 0.08125000000000004}
{'wordA': 'but', 'wordB': 'two', 'similarity': 0.17500000000000004}
{'wordA': 'but', 'wordB': 'pistols', 'similarity': 0.5250000000000001}
{'wordA': 'but', 'wordB': 'on', 'similarity': 0.775}
{'wordA': 'but', 'wordB': 'hip', 'similarity': 0.08125000000000004}
{'wordA': 'but', 'wordB': 'both', 'similarity': 0.55}
{'wordA': 'but', 'wordB': 'are', 'similarity': 0.8875}
{'wordA': 'but', 'wordB': 'gripped', 'similarity': 0.42499999999999993}
{'wordA': 'but', 'wordB': 'i', '

In [6]:
# Milestone 19a — passive logging: append every scored pair to the versioned dataset.
# Idempotent: re-running an unchanged verse appends nothing (see python/score_logging.py).
n_logged = log_scored_pairs(
    similarity_pairs,
    rhyme_candidates,
    DATASET_PATH,
    verse_id=VERSE_ID,
    rhyme_mode=RHYME_MODE,
    detection_mode=DETECTION_MODE,
)
print(f"Logged {n_logged} new scored pairs to {DATASET_PATH}")

Logged 0 new scored pairs to data/scored_pairs.jsonl


In [7]:
# Milestone 11 — panphon similarity spot-check
# Rhyme units below are real IPA sequences from english_mfa alignment of the
# project verse (see Section 4 output). They exercise the new feature-based
# scorer on perfect / slant / non-rhyme pairs.
from python.similarity_engine import rhyme_unit_similarity

spot_check_pairs = [
    # Perfect rhymes — expect 1.0
    ("found",  ["aw", "n", "d"],  "sound",  ["aw", "n", "d"]),
    ("light",  ["aj", "t"],       "night",  ["aj", "t"]),
    ("white",  ["aj", "ʈ"],       "tonight", ["aj", "ʈ"]),

    # Insertion slant (-oud vs -ound: one extra 'n') — expect a clear slant, not 0
    ("crowd",  ["aw", "d"],       "mound",  ["aw", "n", "d"]),

    # Substitution slant (t vs retroflex ʈ) — expect high
    ("light",  ["aj", "t"],       "white",  ["aj", "ʈ"]),

    # Genuinely different — expect low
    ("found",  ["aw", "n", "d"],  "light",  ["aj", "t"]),
    ("the",    ["a"],             "mound",  ["aw", "n", "d"]),
]

print(f"{'Word A':<10} {'Word B':<10} {'Score':>6}  Rhyme units")
print("-" * 55)
for wordA, ruA, wordB, ruB in spot_check_pairs:
    score = rhyme_unit_similarity(ruA, ruB)
    print(f"{wordA:<10} {wordB:<10} {score:>6.3f}  {ruA} | {ruB}")


Word A     Word B      Score  Rhyme units
-------------------------------------------------------
found      sound       1.000  ['aw', 'n', 'd'] | ['aw', 'n', 'd']
light      night       1.000  ['aj', 't'] | ['aj', 't']
white      tonight     1.000  ['aj', 'ʈ'] | ['aj', 'ʈ']
crowd      mound       0.944  ['aw', 'd'] | ['aw', 'n', 'd']
light      white       0.950  ['aj', 't'] | ['aj', 'ʈ']
found      light       0.812  ['aw', 'n', 'd'] | ['aj', 't']
the        mound       0.831  ['a'] | ['aw', 'n', 'd']


---
## Section 6: Words Clustering <a id="clustering"></a>
---

In [8]:
# Clustering by thresholds from the similarity matrix
# get_threshold() returns the correct value for the active RHYME_MODE
cluster_labels, clusters = cluster_rhymes(similarity_matrix, threshold=get_threshold(RHYME_MODE))

print("Clusters:")
print(clusters)

print("\nWord → Cluster Mapping:")
print(cluster_labels)

# Defining expected_clusters and comparing with the currect clustering
expected_clusters = {
    "underground": "A",
    "mound": "A",
    "flight": "B",
    "tonight": "B"
}

for r in rhyme_candidates:
    word = r["end_word"]
    r["expected_cluster"] = expected_clusters.get(word, None)

for r in rhyme_candidates:
    print(r)

# Exact rhymes clusters printout
clusters = cluster_labels

print(clusters)

# Debugging the clustering
debug_rhyme_analysis(rhyme_candidates, clusters)

# Accuracy evaluation
accuracy, mismatches = evaluate_rhyme_detection(rhyme_candidates, clusters)

Clusters:
{'A': ['but', 'as', 'the', 'on', 'are', 'i', 'at', 'a', 'stab', 'dark', 'and', 'my', 'knife', 'off', "i'm", 'now', 'why', 'would', 'about', 'for', 'how', 'bout', 'have'], 'B': ['still', 'explosive', 'clip', 'hip', 'gripped', 'in', 'tip', 'is', 'slit', 'it', 'ultimate', 'chips', 'fit', 'give', 'with', 'width', 'think'], 'C': ['just', 'utmost', 'let', 'they', 'days', 'save', 'fuck', 'may', 'scale', 'up', 'say'], 'D': ['load', 'pistols', 'both', 'hold', 'them', 'shoulder', 'broke', 'to', 'poker', 'were', 'those', 'over', 'kroger', 'so', 'joe', 'oprah'], 'E': ['two'], 'F': ['took'], 'G': ['of', 'your', 'cause', 'fall'], 'H': ['throat', 'cutthroat', 'supposed', 'almost'], 'I': ['where', 'paid', 'wait'], 'J': ['sofas', 'tryna'], 'K': ['that']}

Word → Cluster Mapping:
{'but': 'A', 'as': 'A', 'the': 'A', 'on': 'A', 'are': 'A', 'i': 'A', 'at': 'A', 'a': 'A', 'stab': 'A', 'dark': 'A', 'and': 'A', 'my': 'A', 'knife': 'A', 'off': 'A', "i'm": 'A', 'now': 'A', 'why': 'A', 'would': 'A', 'a

In [9]:
print(clusters)

{'but': 'A', 'as': 'A', 'the': 'A', 'on': 'A', 'are': 'A', 'i': 'A', 'at': 'A', 'a': 'A', 'stab': 'A', 'dark': 'A', 'and': 'A', 'my': 'A', 'knife': 'A', 'off': 'A', "i'm": 'A', 'now': 'A', 'why': 'A', 'would': 'A', 'about': 'A', 'for': 'A', 'how': 'A', 'bout': 'A', 'have': 'A', 'still': 'B', 'explosive': 'B', 'clip': 'B', 'hip': 'B', 'gripped': 'B', 'in': 'B', 'tip': 'B', 'is': 'B', 'slit': 'B', 'it': 'B', 'ultimate': 'B', 'chips': 'B', 'fit': 'B', 'give': 'B', 'with': 'B', 'width': 'B', 'think': 'B', 'just': 'C', 'utmost': 'C', 'let': 'C', 'they': 'C', 'days': 'C', 'save': 'C', 'fuck': 'C', 'may': 'C', 'scale': 'C', 'up': 'C', 'say': 'C', 'load': 'D', 'pistols': 'D', 'both': 'D', 'hold': 'D', 'them': 'D', 'shoulder': 'D', 'broke': 'D', 'to': 'D', 'poker': 'D', 'were': 'D', 'those': 'D', 'over': 'D', 'kroger': 'D', 'so': 'D', 'joe': 'D', 'oprah': 'D', 'two': 'E', 'took': 'F', 'of': 'G', 'your': 'G', 'cause': 'G', 'fall': 'G', 'throat': 'H', 'cutthroat': 'H', 'supposed': 'H', 'almost': 

In [10]:
# Milestone 9 — full pipeline regression check
from python.similarity_engine import compute_similarity_pairs, build_similarity_matrix
from python.clustering import cluster_rhymes, get_threshold
from python.html_generation import filter_clusters

similarity_pairs = compute_similarity_pairs(rhyme_candidates, rhyme_mode=RHYME_MODE)
similarity_matrix = build_similarity_matrix(rhyme_candidates, similarity_pairs)
word_to_cluster, clusters = cluster_rhymes(similarity_matrix, threshold=get_threshold(RHYME_MODE))

passing_labels = filter_clusters(word_to_cluster, rhyme_candidates)

passing = {}
for entry in rhyme_candidates:
    word = entry["end_word"]
    label = word_to_cluster.get(word)
    if label in passing_labels:
        passing.setdefault(label, [])
        if word not in passing[label]:
            passing[label].append(word)

print("Passing clusters:")
for label, members in sorted(passing.items()):
    print(f"  {label}: {members}")

Passing clusters:
  A: ['but', 'as', 'the', 'on', 'are', 'i', 'at', 'a', 'stab', 'dark', 'and', 'my', 'knife', 'off', "i'm", 'now', 'why', 'would', 'about', 'for', 'how', 'bout', 'have']
  B: ['still', 'explosive', 'with', 'clip', 'hip', 'gripped', 'width', 'in', 'tip', 'is', 'slit', 'it', 'ultimate', 'chips', 'fit', 'give', 'think']
  C: ['just', 'utmost', 'let', 'they', 'days', 'save', 'fuck', 'may', 'scale', 'up', 'say']
  D: ['load', 'pistols', 'both', 'hold', 'them', 'shoulder', 'broke', 'to', 'poker', 'were', 'those', 'over', 'kroger', 'so', 'joe', 'oprah']
  G: ['of', 'your', 'cause', 'fall']
  H: ['throat', 'cutthroat', 'supposed', 'almost']
  I: ['where', 'paid', 'wait']


---
## Section 7: HTML Generation <a id="html-generation"></a>
---

In [11]:
clusters = clusters = cluster_labels

generate_rhyme_html(rhyme_candidates, clusters, detection_mode=DETECTION_MODE, debug=True)

[HTML] Passing clusters: ['A', 'B', 'C', 'D', 'G', 'H', 'I']
[HTML] Blocked clusters: ['E', 'F', 'J', 'K']
HTML file generated: generated_html/rhyme_visualization.html


---
## Section 8: Syllable Engine (boundary-free) — primary view <a id="syllable-engine"></a>
---

Milestone 13. The **syllable** — not the word — is the atomic unit. Syllable
units cluster **freely across word and line boundaries**, so a long word can
participate in a scheme through any of its syllables, and a multi-syllable word
can wear several colours (one per syllable's cluster). This runs **in parallel**
to the word-based path in Sections 4–7, which is left untouched (it still feeds
the Milestone 19a pair-logging dataset).

In [12]:
# Split every word into syllable units (one comparison unit per syllable), each
# carrying a back-pointer to its source word + phoneme range for colouring.
syllable_units = extract_syllable_candidates(
    INPUT_TXT_PATH,
    word_to_phonemes,
    detection_mode=DETECTION_MODE,
)

for u in syllable_units:
    print(u["unit_id"], u["source_word"], "".join(u["phonemes"]))

0:0:0 but bat
0:1:0 still stʲiɫ
0:2:0 as as
0:3:0 explosive ɛks
0:3:1 explosive plo
0:3:2 explosive siv
0:4:0 with ʋɪd̪
0:5:0 just dʒɐst
0:6:0 load ləwd
0:7:0 the ða
0:8:0 clip cʎip
1:0:0 two ʈʉː
1:1:0 pistols pɪs
1:1:1 pistols ʈəlz
1:2:0 on an
1:3:0 hip çip
1:4:0 both bəwθ
1:5:0 are a
1:6:0 gripped ɟɹipt
2:0:0 i aj
2:1:0 hold həwɫd
2:2:0 them ðəm
2:3:0 at at
2:4:0 shoulder ʃəwɫ
2:4:1 shoulder də
2:5:0 width ʋɪd̪
3:0:0 took ʈʊk
3:1:0 a a
3:2:0 stab stab
3:3:0 in in
3:4:0 the ða
3:5:0 dark dak
3:6:0 and an
3:7:0 broke bɹəwk
3:8:0 the ða
3:9:0 tip ʈɪp
4:0:0 of ɒv
4:1:0 my mɑ
4:2:0 knife najf
4:3:0 off ɑf
4:4:0 but bat
4:5:0 your jɒɹ
4:6:0 throat t̪ɹoːʈ
4:7:0 is iz
4:8:0 slit sʎit
5:0:0 cause kɒz
5:1:0 i'm ajm
5:2:0 cutthroat kə
5:2:1 cutthroat t̪ɹoːʈ
5:3:0 to ə
5:4:0 the ða
5:5:0 utmost ɐ
5:5:1 utmost tməwst
5:6:0 with ʋɪd̪
5:7:0 it it
6:0:0 the ða
6:1:0 ultimate ɐɫ
6:1:1 ultimate tə
6:1:2 ultimate mʲɪt
7:0:0 i aj
7:1:0 just dʒɐst
7:2:0 let lɛt
7:3:0 the ða
7:4:0 poker poː
7:4:1 poker kə

In [13]:
# Score every pair of syllable units (role-aware panphon scorer: onset weight 0,
# nucleus full, coda discounted) and build the unit_id-keyed similarity matrix.
syllable_pairs = compute_syllable_pairs(syllable_units)
syllable_matrix = build_syllable_matrix(syllable_units, syllable_pairs)

print(f"{len(syllable_units)} syllable units, {len(syllable_pairs)} pairs scored")

130 syllable units, 8385 pairs scored


In [14]:
# Cluster the syllable units (average-linkage, carried at threshold 0.7).
# Clustering is keyed on each syllable's unit_id, so syllables group freely
# across word and line boundaries.
syllable_labels, syllable_clusters = cluster_rhymes(
    syllable_matrix, threshold=SYLLABLE_THRESHOLD
)

for label in sorted(syllable_clusters):
    members = [
        f'{u["source_word"]}·{u["syllable_index"]}'
        for u in syllable_units
        if syllable_labels[u["unit_id"]] == label
    ]
    print(f"{label}: {members}")

A: ['but·0', 'as·0', 'just·0', 'the·0', 'pistols·1', 'on·0', 'are·0', 'at·0', 'a·0', 'stab·0', 'the·0', 'dark·0', 'and·0', 'the·0', 'my·0', 'off·0', 'but·0', 'your·0', 'cause·0', 'the·0', 'the·0', 'just·0', 'the·0', 'fall·0', 'are·0', 'at·0', 'kroger·0', 'would·0', 'a·0', 'about·0', 'about·1', 'for·0', 'bout·0', 'as·0', 'have·0', 'the·0', 'cause·0', 'the·0', 'but·0', 'cause·0', 'almost·0']
B: ['still·0', 'explosive·2', 'with·0', 'clip·0', 'pistols·0', 'hip·0', 'gripped·0', 'width·0', 'in·0', 'tip·0', 'is·0', 'slit·0', 'utmost·0', 'with·0', 'it·0', 'ultimate·0', 'ultimate·2', 'let·0', 'chips·0', 'they·0', 'fit·0', 'days·0', 'in·0', 'save·0', 'give·0', 'fuck·0', 'think·0', 'may·0', 'scale·0', 'is·0', 'up·0', 'they·0', 'say·0']
C: ['explosive·0', 'load·0', 'both·0', 'hold·0', 'them·0', 'shoulder·0', 'shoulder·1', 'broke·0', 'cutthroat·0', 'to·0', 'utmost·1', 'ultimate·1', 'poker·1', 'were·0', 'supposed·0', 'to·0', 'those·0', 'over·0', 'over·1', 'kroger·1', 'so·0', 'joe·0', 'oprah·0', 'opr

In [15]:
# Per-syllable colour rendering: each syllable wears its cluster's colour, so a
# multi-syllable word can carry several colours (the "Verse DNA" texture).
generate_syllable_html(syllable_units, syllable_labels, debug=True)

[SYL HTML] Passing clusters: ['A', 'B', 'C', 'D', 'F', 'H', 'I']
[SYL HTML] Blocked clusters: ['E', 'G', 'J', 'K']
Syllable HTML file generated: generated_html/syllable_visualization.html


'<html>\n<head>\n<title>Verse DNA Syllable Visualization</title>\n<style>\nbody { font-family: Arial; font-size: 18px; line-height: 1.8; }\n.syl-A { background-color: #ffcccc; border-radius: 3px; padding: 1px 2px; }\n.syl-B { background-color: #cce5ff; border-radius: 3px; padding: 1px 2px; }\n.syl-C { background-color: #ccffcc; border-radius: 3px; padding: 1px 2px; }\n.syl-D { background-color: #ffe6cc; border-radius: 3px; padding: 1px 2px; }\n.syl-F { background-color: #ffffcc; border-radius: 3px; padding: 1px 2px; }\n.syl-H { background-color: #e6ccff; border-radius: 3px; padding: 1px 2px; }\n.syl-I { background-color: #ffd6e7; border-radius: 3px; padding: 1px 2px; }\n</style>\n</head>\n<body>\n<span class="syl-A">But</span> <span class="syl-B">still</span> <span class="syl-A">as</span> <span class="syl-C">exp</span><span class="syl-D">los</span><span class="syl-B">ive</span> <span class="syl-B">with,</span> <span class="syl-A">just</span> <span class="syl-C">load</span> <span class=